In [2]:
import torch
import torch.nn as nn
import numpy as np

In [ ]:
class MyMHA(nn.Module):
    def __init__(self, D_1, D_2, D_v, D_qk, H):
        super().__init__()
        self.D_1 = D_1
        self.D_2 = D_2
        self.D_v = D_v
        self.D_qk = D_qk
        self.H = H
        
        self.W_Q = nn.Linear(in_features=D_1, out_features=H*D_qk, bias=False)
        self.W_K = nn.Linear(in_features=D_2, out_features=H*D_qk, bias=False)
        self.W_V = nn.Linear(in_features=D_2, out_features=H*D_v, bias=False)
        self.W_O = nn.Linear(in_features=H*D_v, out_features=D_1, bias=False)

    def forward(self, x, y):
        B = x.shape[0]
        L_1 = x.shape[1]
        L_2 = y.shape[1]

        q = self.W_Q(x) # (B, L_1, H*D_qk)
        k = self.W_K(y) # (B, L_2, H*D_qk)
        v = self.W_V(y) # (B, L_2, H*D_v)

        q = q.reshape(B, L_1, self.H, self.D_qk)
        k = k.reshape(B, L_2, self.H,self.D_qk)
        v = v.reshape(B, L_2,self.H,self.D_v)

        q = q.permute(0,2,1,3) # (B,H,L_1,D_qk)
        k = k.permute(0,2,1,3) # (B,H,L_2,D_qk)
        v = v.permute(0,2,1,3) # (B,H,L_2,D_v)

        logits = q @ k.transpose(-2, -1) / self.D_qk**0.5**0.5 # (B, H, L_1, L_2)
        alpha = torch.softmax(logits, dim=-1) # (B, H, L_1, L_2)
        o = alpha @ v # (B, H, L_1, D_v)
        o = o.permute(0,2,1,3) # (B, L_1, H, D_v)
        o = o.reshape(B, L_1, self.H*self.D_v)
        res = self.W_O(o) # (B, L_1, D1)
        return res

In [ ]:
class MyGQA(nn.Module):
    def __init__(self, D_1, D_2, D_v, D_qk, H, G):
        super().__init__()
        self.D_1 = D_1
        self.D_2 = D_2
        self.D_v = D_v
        self.D_qk = D_qk
        self.H = H
        self.G = G
        
        self.W_Q = nn.Linear(in_features=D_1, out_features=H*D_qk, bias=False)
        self.W_K = nn.Linear(in_features=D_2, out_features=G*D_qk, bias=False)
        self.W_V = nn.Linear(in_features=D_2, out_features=G*D_v, bias=False)
        self.W_O = nn.Linear(in_features=H*D_v, out_features=D_1, bias=False)

    def forward(self, x, y):
        B = x.shape[0]
        L_1 = x.shape[1]
        L_2 = y.shape[1]

        q = self.W_Q(x) # (B, L_1, H*D_qk)
        k = self.W_K(y) # (B, L_2, G*D_qk)
        v = self.W_V(y) # (B, L_2, G*D_v)

        q = q.reshape(B, L_1, self.H//self.G, self.G, self.D_qk)
        k = k.reshape(B, L_2, 1, self.G,self.D_qk)
        v = v.reshape(B, L_2, 1, self.G,self.D_v)

        q = q.permute(0,2,3,1,4) # (B,H/G, G,L_1,D_qk)
        k = k.permute(0,2,3,1,4) # (B,1, G,L_2,D_qk)
        v = v.permute(0,2,3,1,4) # (B,1, G,L_2,D_v)

        logits = q @ k.transpose(-2, -1) / self.D_qk**0.5 # (B, H/G, G, L_1, L_2)
        alpha = torch.softmax(logits, dim=-1) # (B, H/G, G, L_1, L_2)
        o = alpha @ v # (B, H/G, G, L_1, D_v)
        o = o.permute(0,3,1,2,4) # (B, L_1, H/G, G, D_v)
        o = o.reshape(B, L_1, self.H*self.D_v)
        res = self.W_O(o) # (B, L_1, D1)
        return res

In [16]:
def GQA_2_MLA(W_M_GQA):
    r, D = W_M_GQA.shape
    W_M_GQA_tilde = np.concatenate(([W_M_GQA] * (D//r)))

    u, s, v = np.linalg.svd(W_M_GQA_tilde)
    W_UM_MLA = u[:, :r]
    W_DKV_MLA = s.reshape(-1, 1)[:r, :] * v[:r, :]
    print(f"Shape of W_UK_MLA: {W_UM_MLA.shape}")
    print(f"Shape of W_DKV_MLA: {W_DKV_MLA.shape}")
    mse = np.mean((W_M_GQA_tilde - W_UM_MLA @ W_DKV_MLA)**2)
    print(mse)

    return W_M_GQA_tilde.shape

GQA_2_MLA(np.random.randn(4, 24))

Shape of W_UK_MLA: (24, 4)
Shape of W_DKV_MLA: (4, 24)
8.824950069997286e-31


(24, 24)